<a href="https://colab.research.google.com/github/CHINGUYEN-alt/Yolov5-object-detection/blob/main/Yolov5detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 1: Clone và cài đặt YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Cài đặt requirements
!pip install -r requirements.txt

print("✅ YOLOv5 installed successfully")

Cloning into 'yolov5'...
remote: Enumerating objects: 17822, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 17822 (delta 16), reused 6 (delta 6), pack-reused 17794 (from 2)
Receiving objects: 100% (17822/17822), 17.01 MiB | 19.70 MiB/s, done.
Resolving deltas: 100% (12149/12149), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
✅ YOLOv5 installed successfully


In [ ]:
# Cell: CHUYỂN ĐỔI DATASET TỪ CẤU TRÚC PHÂN LỚP SANG CẤU TRÚC PHẲNG
import os
import shutil
from glob import glob
from tqdm import tqdm

# Đường dẫn
BASE_PATH = "/content/drive/MyDrive/datasheet"
DEST_PATH = "/content/drive/MyDrive/datasheet_yolo"  # Tên mới

print("🔄 BẮT ĐẦU CHUYỂN ĐỔI DATASET...")
print("="*60)

def convert_split(split):
    src_images = f"{BASE_PATH}/images/{split}"
    src_labels = f"{BASE_PATH}/labels/{split}"

    dst_images = f"{DEST_PATH}/images/{split}"
    dst_labels = f"{DEST_PATH}/labels/{split}"

    os.makedirs(dst_images, exist_ok=True)
    os.makedirs(dst_labels, exist_ok=True)

    # Lấy danh sách các lớp (thư mục con)
    classes = [d for d in os.listdir(src_images)
               if os.path.isdir(os.path.join(src_images, d))]

    total_images = 0
    total_labels = 0

    for cls in tqdm(classes, desc=f"Xử lý {split}"):
        # Copy ảnh
        img_files = glob(os.path.join(src_images, cls, "*.*"))
        for img in img_files:
            ext = os.path.splitext(img)[1]
            if ext.lower() in ['.jpg', '.jpeg', '.png']:
                new_name = f"{cls}_{os.path.basename(img)}"
                shutil.copy2(img, os.path.join(dst_images, new_name))
                total_images += 1

        # Copy label
        label_files = glob(os.path.join(src_labels, cls, "*.txt"))
        for lbl in label_files:
            new_name = f"{cls}_{os.path.basename(lbl)}"
            shutil.copy2(lbl, os.path.join(dst_labels, new_name))
            total_labels += 1

    return total_images, total_labels

# Chuyển đổi train và val
train_img, train_lbl = convert_split('train')
val_img, val_lbl = convert_split('val')
test_img, test_lbl = convert_split('test')

print("="*60)
print("📊 KẾT QUẢ CHUYỂN ĐỔI:")
print(f"   - Train: {train_img} ảnh, {train_lbl} labels")
print(f"   - Val: {val_img} ảnh, {val_lbl} labels")
print(f"   - Test: {test_img} ảnh, {test_lbl} labels")
print("="*60)
print(f"✅ Dataset đã được chuyển đổi thành công!")
print(f"📁 Đường dẫn: {DEST_PATH}")

🔄 BẮT ĐẦU CHUYỂN ĐỔI DATASET...


Xử lý test: 100%|██████████| 8/8 [04:22<00:00, 32.77s/it]

📊 KẾT QUẢ CHUYỂN ĐỔI:
   - Train: 18209 ảnh, 18209 labels
   - Val: 2143 ảnh, 2143 labels
   - Test: 1070 ảnh, 1070 labels
✅ Dataset đã được chuyển đổi thành công!
📁 Đường dẫn: /content/drive/MyDrive/datasheet_yolo


In [ ]:
# Cell: TẠO FILE data.yaml
%%writefile /content/drive/MyDrive/datasheet_yolo/data.yaml
train: /content/drive/MyDrive/datasheet_yolo/images/train
val: /content/drive/MyDrive/datasheet_yolo/images/val
test: /content/drive/MyDrive/datasheet_yolo/images/test

nc: 8
names: ['Center', 'Donut', 'Scratch', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random']

Writing /content/drive/MyDrive/datasheet_yolo/data.yaml


In [ ]:
# Cell: KIỂM TRA DATASET
import os

train_path = "/content/drive/MyDrive/datasheet_yolo/images/train"
val_path = "/content/drive/MyDrive/datasheet_yolo/images/val"

train_count = len(os.listdir(train_path))
val_count = len(os.listdir(val_path))

print(f"✅ Train images: {train_count}")
print(f"✅ Val images: {val_count}")
print(f"✅ Total: {train_count + val_count} images")

if train_count == 18209 and val_count == 2143:
    print(" Dataset OK! Sẵn sàng train YOLOv5!")
else:
    print(f"⚠️ Dataset không khớp! Train: {train_count}/18209, Val: {val_count}/2143")

✅ Train images: 18209
✅ Val images: 2143
✅ Total: 20352 images
🎉 Dataset OK! Sẵn sàng train YOLOv5!


In [ ]:
# Cell: CÀI ĐẶT YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

print("✅ YOLOv5 installed")

Cloning into 'yolov5'...
remote: Enumerating objects: 17842, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 17842 (delta 23), reused 6 (delta 6), pack-reused 17806 (from 3)
Receiving objects: 100% (17842/17842), 16.99 MiB | 18.49 MiB/s, done.
Resolving deltas: 100% (12164/12164), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
✅ YOLOv5 installed


In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data /content/drive/MyDrive/datasheet_yolo/data.yaml \
    --weights yolov5s.pt \
    --project /content/drive/MyDrive/yolo_weights \
    --name yolov5_wafer \
    --exist-ok \
    --device 0 \
    --workers 2

/content/yolov5
terminate called after throwing an instance of 'pybind11::error_already_set'
  what():  KeyboardInterrupt: <MESSAGE UNAVAILABLE DUE TO ANOTHER EXCEPTION>

MESSAGE UNAVAILABLE DUE TO EXCEPTION: KeyboardInterrupt: <EMPTY MESSAGE>


In [ ]:
%cd /content/yolov5
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data /content/drive/MyDrive/datasheet_yolo/data.yaml \
    --weights /content/drive/MyDrive/yolo_weights/yolov5_wafer/weights/last.pt \
    --project /content/drive/MyDrive/yolo_weights \
    --name yolov5_wafer \
    --exist-ok \
    --device 0 \
    --workers 8 \
    --resume /content/drive/MyDrive/yolo_weights/yolov5_wafer/weights/last.pt

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
      97/99      4.55G    0.01691    0.01144   0.002358         47        640:  82% 933/1139 [08:09<01:28,  2.32it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      97/99      4.55G    0.01692    0.01144   0.002364         44        640:  82% 934/1139 [08:10<01:38,  2.09it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      97/99      4.55G    0.01691    0.01144   0.002361         31        640:  82% 935/1139 [08:10<01:23,  2.46it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      97/99      4.55G

In [ ]:
# Kiểm tra số lượng ảnh trong dataset
import os

train_path = "/content/drive/MyDrive/datasheet_yolo/images/train"
val_path = "/content/drive/MyDrive/datasheet_yolo/images/val"

train_count = len(os.listdir(train_path))
val_count = len(os.listdir(val_path))

print(f"✅ Train images: {train_count}")
print(f"✅ Val images: {val_count}")
print(f"✅ Total: {train_count + val_count} images")

✅ Train images: 18209
✅ Val images: 2143
✅ Total: 20352 images


In [ ]:
# CHẠY YOLOv5 TRÊN TEST SET
%cd /content/yolov5

!python val.py \
    --data /content/drive/MyDrive/datasheet_yolo/data.yaml \
    --weights /content/drive/MyDrive/yolo_weights/yolov5_wafer/weights/best.pt \
    --img 640 \
    --batch 16 \
    --device 0 \
    --project /content/drive/MyDrive/yolo_weights \
    --name yolov5_test_1070 \
    --exist-ok \
    --conf-thres 0.001 \
    --iou-thres 0.6 \
    --task test  #

/content/yolov5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
val: data=/content/drive/MyDrive/datasheet_yolo/data.yaml, weights=['/content/drive/MyDrive/yolo_weights/yolov5_wafer/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=0, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/yolo_weights, name=yolov5_test_1070, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-461-gda75ed9d Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7031701 parameters, 0 gradients, 15.8 GFLOPs
100% 755k/755k [00:00<00:00, 